# Imports and preprocessing

In [49]:
import csv
import numpy as np
from pathlib import Path
import sys, os
import importlib
sys.path.append(os.path.join(os.getcwd(), '..'))

from utils import eda_helpers, all_functions, cv, sampling_methods
import implementations

importlib.reload(eda_helpers)
importlib.reload(all_functions)
importlib.reload(cv)
importlib.reload(sampling_methods)
importlib.reload(implementations)

from utils.eda_helpers import clean_data, read_csv_numeric
from utils.all_functions import sigmoid
from implementations import (
    logistic_regression,
    reg_logistic_regression,
    weighted_logistic_regression,
)

file_path = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_FOLDER = os.path.join(file_path, '..', 'dataset')

In [2]:
x_indices, x_data = read_csv_numeric(os.path.join(DATA_FOLDER, "x_train.csv"))
y_indices, y_data = read_csv_numeric(os.path.join(DATA_FOLDER, "y_train.csv"))

In [5]:
final_feature_set = ['_CASTHM1', 'FLUSHOT6', 'METVL11_', '_PASTRNG', 'EDUCA', 'CHCCOPD1', 'DRADVISE', 
                     'CHCKIDNY', 'ARTHSOCL', 'MENTHLTH', 'SMOKE100', 'DOCTDIAB', 'DIABAGE2', 'VETERAN3', 
                     'CHKHEMO3', 'SEX', '_DRNKWEK', 'JOINPAIN', 'MARITAL', 'EXERANY2', '_PACAT1', 'INSULIN', 
                     'DIFFALON', 'FRUIT1', 'QLACTLM2', 'BLIND', 'PADUR2_', 'PAFREQ2_', 'LMTJOIN3', 'CHOLCHK', 
                     'POORHLTH', 'DIABETE3', '_FRUTSUM', 'PERSDOC2', 'EXRACT21', 'FEETCHK', '_VEGESUM', 'PNEUVAC3', 
                     'BPHIGH4', 'GENHLTH', 'EXRACT11', '_BMI5', 'DECIDE', '_ASTHMS1', '_MRACE1', '_LMTSCL1', 'ORNGDAY_', 
                     'MSCODE', 'PHYSHLTH', 'TOLDHI2', 'PADUR1_', 'DIFFWALK', 'BEANDAY_', 'CHCSCNCR', 'PAFREQ1_', '_AIDTST3', 
                     'FLSHTMY2', 'PA1VIGM_', 'METVL21_', 'SXORIENT', 'ARTHDIS2', 'PA1MIN_', 'MAXVO2_', '_RACE_G1', 'LONGWTCH', 
                     'SCNTWRK1', 'FEETCHK2', 'EMPLOY1', 'BLOODCHO', 'USENOW3', 'DRNKANY5', '_PAREC1', 'DIFFDRES', 'USEEQUIP', 
                     'ADDEPEV2', '_AGE80', 'HTM4', 'INCOME2', 'CVDSTRK3', 'CHCOCNCR', 'MEDCOST', 'PREDIAB1', 'BPMEDS', 'HAVARTH3', 
                     'CHECKUP1', 'TRNSGNDR', 'CHILDREN']

In [6]:
del x_indices["Id"]
x_data = clean_data(x_data, x_indices)

In [7]:
# --- Keep only the columns of interest ---
x_data_interested = x_data[:, [x_indices[col] for col in final_feature_set]]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Drop columns that are all NaN ---
valid_cols = ~np.isnan(x_data_interested).all(axis=0)
x_data_interested = x_data_interested[:, valid_cols]
interesting_columns = [col for col, keep in zip(final_feature_set, valid_cols) if keep]
x_indices_interested = {col: i for i, col in enumerate(final_feature_set)}

# --- Impute remaining NaNs with column mean ---
col_mean = np.nanmean(x_data_interested, axis=0)
inds = np.where(np.isnan(x_data_interested))
x_data_interested[inds] = np.take(col_mean, inds[1])

# --- Standardize once ---
x_data_interested = (x_data_interested - np.mean(x_data_interested, axis=0)) / np.std(x_data_interested, axis=0)

# --- Prepare y ---
y = y_data[:,1].copy()
y[np.isnan(y)] = np.nanmean(y)
y = (y + 1) / 2  # convert -1,1 → 0,1
y = y.flatten()

# --- Checks ---
print("Unique labels:", np.unique(y))
print("Any NaN in features?", np.any(np.isnan(x_data_interested)))
print("Min std across features:", np.min(np.std(x_data_interested, axis=0)))
print("Label distribution (mean):", np.mean(y))

Unique labels: [0. 1.]
Any NaN in features? False
Min std across features: 0.9999999999999998
Label distribution (mean): 0.08830207079403295


# Evaluation

In [50]:
from utils.cv import (
    cross_validate,
    train_logreg,
    predict,
    compute_training_error,
)
from utils.sampling_methods import (
    undersample,
    oversample,
    smote
)


In [51]:
sampling_methods = {
    "none": None,
    "undersample": undersample,
    "oversample": oversample,
    "smote": smote
}
sampling_kwargs = {
    "undersample": {"ratio": 2.0, "seed": 42},
    "oversample": {"ratio": 3.0, "seed": 42},
    "smote": {"k": 5, "ratio": 3.0, "seed": 42},
}

all_results = []

In [ ]:
# --- Cross-validation for all logistic regression variants ---

for kind in ["logistic", "reg_logistic", "w_logistic"]:
    print(f"\n=== MODEL: {kind.upper()} ===")

    for s_name, s_func in sampling_methods.items():
        print(f"\n--- Sampling: {s_name.upper()} ---")

        # cross-validation
        results = cross_validate(
            model_fn=lambda y, X: train_logreg(
                y, X,
                kind="logistic",
                gamma=0.01,
                max_iters=500,
                verbose=False
            ),
            X=x_data_interested,
            y=y,
            k_folds=5,
            seed=42,
            sampling_fn=s_func,
            sampling_kwargs=sampling_kwargs.get(s_name, None),
            predict_fn=lambda X, w: (predict(X, w), predict(X, w, return_prob=True)),
            metric_fn=compute_training_error,
            use_predictions=False,
            verbose=True
        )

        # store results
        model_result = {"model": kind, "sampling": s_name, **results}
        all_results.append(model_result)
        print("Average CV metrics:", results)



=== MODEL: LOGISTIC ===

--- Sampling: NONE ---
Fold 1/5 → error:0.350, accuracy:0.650, precision:0.187, recall:0.898, f1:0.310, roc_auc:0.152
Fold 2/5 → error:0.348, accuracy:0.652, precision:0.190, recall:0.896, f1:0.314, roc_auc:0.153
Fold 3/5 → error:0.347, accuracy:0.653, precision:0.191, recall:0.901, f1:0.315, roc_auc:0.147


In [ ]:
# print("\n=== MODEL: LOGISTIC REGRESSION ===")

# for s_name, s_func in sampling_methods.items():
#     print(f"\n--- Sampling: {s_name.upper()} ---")

#     # Cross-validation
#     results = cross_validate(
#         model_fn=lambda y, X: train_logreg(
#             y, X,
#             kind="logistic",
#             gamma=0.01,
#             max_iters=500,
#             verbose=False
#         ),
#         X=x_data_interested,
#         y=y,
#         k_folds=5,
#         seed=42,
#         sampling_fn=s_func,
#         sampling_kwargs=sampling_kwargs.get(s_name, None),
#         predict_fn=lambda X, w: (predict(X, w), predict(X, w, return_prob=True)),
#         metric_fn=compute_training_error,
#         use_predictions=False,
#         verbose=True
#     )

#     model_result = {"model": "logistic", "sampling": s_name, **results}
#     all_results.append(model_result)
#     print("Average CV metrics:", results)


=== MODEL: LOGISTIC REGRESSION ===

--- Sampling: NONE ---


UnboundLocalError: cannot access local variable 'w' where it is not associated with a value

In [ ]:
print("\n=== MODEL: REGULARIZED LOGISTIC REGRESSION ===")

for s_name, s_func in sampling_methods.items():
    print(f"\n--- Sampling: {s_name.upper()} ---")

    results = cross_validate(
        model_fn=lambda X, y: train_logreg(
            y, X,
            kind="reg_logistic",
            gamma=0.01,
            lambda_=0.001,
            max_iters=500,
            verbose=False
        ),
        X=x_data_interested,
        y=y,
        k_folds=5,
        seed=42,
        sampling_fn=s_func,
        sampling_kwargs=sampling_kwargs.get(s_name, None),
        predict_fn=lambda X, w: (predict(X, w), predict(X, w, return_prob=True)),
        metric_fn=compute_training_error,
        use_predictions=False,
        verbose=True
    )

    model_result = {"model": "reg_logistic", "sampling": s_name, **results}
    all_results.append(model_result)
    print("Average CV metrics:", results)


# disregard

In [3]:
def cross_validate_logreg(X, y, model_kind="logistic",
                          gamma=0.01, lambda_=0.0, max_iters=500,
                          tol=1e-4, k_folds=5, seed=42,
                          verbose=True):
    """
    Simple and memory-safe k-fold cross-validation for logistic regression variants.
    No sampling, no nested lambdas — just pure folds.

    model_kind: "logistic" | "reg_logistic" | "w_logistic"
    """

    from utils.cv import train_logreg, predict, compute_training_error

    rng = np.random.default_rng(seed)
    n = len(y)
    indices = rng.permutation(n)
    folds = np.array_split(indices, k_folds)

    fold_metrics = []

    for i, val_idx in enumerate(folds):
        train_idx = np.setdiff1d(indices, val_idx)
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # train
        w, _ = train_logreg(
            y_train, X_train,
            kind=model_kind,
            gamma=gamma,
            lambda_=lambda_,
            max_iters=max_iters,
            tol=tol,
            verbose=False
        )

        # predict
        y_pred, y_proba = predict(X_val, w), predict(X_val, w, return_prob=True)

        # evaluate
        metrics = compute_training_error(y_val, y_pred, y_proba)
        fold_metrics.append(metrics)

        if verbose:
            msg = f"Fold {i+1}/{k_folds} → " + ", ".join([f"{k}:{v:.3f}" for k,v in metrics.items()])
            print(msg)

    # average results
    avg_metrics = {m: np.mean([fold[m] for fold in fold_metrics]) for m in fold_metrics[0]}
    return avg_metrics

In [12]:
def train_logreg(y, tx, kind="logistic",
                 gamma=0.01, lambda_=0.0, max_iters=1000,
                 tol=None, w_init=None, verbose=True):
    """
    Unified training wrapper for logistic regression variants:
    - Standard logistic regression
    - Regularized logistic regression
    - Weighted logistic regression
    """
    n_features = tx.shape[1]
    if w_init is None:
        w_init = np.zeros((n_features, 1))
    y = y.flatten()


    if kind == "logistic":
        # Standard logistic regression (5 args)
        w, losses = logistic_regression(y, tx, w_init, max_iters, gamma)

    elif kind == "reg_logistic":
        # Regularized logistic regression (6 args)
        w, losses = reg_logistic_regression(y, tx, lambda_, w_init, max_iters, gamma)

    elif kind == "w_logistic":
        # Weighted logistic regression (7 args)
        w, losses = weighted_logistic_regression(y, tx, w_init, max_iters, gamma, lambda_, tol)

    else:
        raise ValueError("kind must be 'logistic', 'reg_logistic', or 'w_logistic'")

    if verbose:
        print(f"[{kind}] Training finished — final loss: {losses[-1]:.6f}")

    return w, losses[-1]



In [ ]:
from implementations import calculate_loss, calculate_gradient
from utils.all_functions import sigmoid

def calculate_gradient(y, tx, w):
    """compute the gradient of loss.

    Args:
        y:  shape=(N, 1)
        tx: shape=(N, D)
        w:  shape=(D, 1)

    Returns:
        the gradient grad, a vector of shape (D, 1)
    """
    y = y.reshape(-1, 1)  
    grad = tx.T @ (sigmoid(tx @ w) - y) / y.shape[0]
    return grad

X_small = x_data_interested[:1000]
y_small = y[:1000]
w = np.zeros((X_small.shape[1], 1))

print("X_small:", X_small.shape)
print("y_small:", y_small.shape)
print("w:", w.shape)

# Try loss and gradient
loss = calculate_loss(y_small, X_small, w)
grad = calculate_gradient(y_small, X_small, w)

print("loss:", loss)
print("grad shape:", grad.shape)


X_small: (1000, 87)
y_small: (1000,)
w: (87, 1)
loss: 693.1471805599454
grad shape: (87, 1)


In [ ]:
from implementations import calculate_loss, sigmoid
import numpy as np

X_small = x_data_interested[:5000]
y_small = y[:5000]
w = np.zeros((X_small.shape[1], 1))

print("Shapes -> X:", X_small.shape, "y:", y_small.shape, "w:", w.shape)

loss = calculate_loss(y_small, X_small, w)
print("Loss computed successfully:", loss)


Shapes -> X: (5000, 87) y: (5000,) w: (87, 1)
Loss computed successfully: 3465.7359027997272


In [ ]:
print("Shapes before CV:")
print("x_data_interested:", x_data_interested.shape)
print("y:", y.shape)
print("y ndim:", y.ndim)


Shapes before CV:
x_data_interested: (328135, 87)
y: (328135,)
y ndim: 1
